## Part 3: Features, split and preprocessing (P3)

### Section 3.1: Setup

In [ ]:
# LOCAL SETUP (removed automatically when the notebooks are merged)
import sys
sys.path.append("../src")
from f1_clean import load_clean, SEED

In [ ]:
clean = load_clean()
base     = clean["model_base"]   # pre-race columns + target 'podium'
outcomes = clean["outcomes"]     # post-race columns — NEVER use as features
tables   = clean["tables"]       # all other cleaned tables

base.shape, outcomes.shape, list(tables.keys())

### Section 3.2: Base table sanity check

p1 cleaned mode_base with 0 duplicated rows for (raceId, driverId)

our target 'podium' is binary int8 with values [0,1]. the overall podium rate is around 13.7% which matches the expected 3/field-size ratio.

missing values are present in the qualifying-related columns q1_s, q2_S, q3_s, qual_position.

this is what we want to target when choosing the start year since the f1 knockout started after 2006.

In [ ]:
base.head()

In [ ]:
base.dtypes

In [ ]:
# Shape of the cleaned base table
print("base shape:", base.shape)

# Grain: one row per (raceId, driverId)
dupes = base.duplicated(["raceId", "driverId"]).sum()
print("duplicated (raceId, driverId):", dupes)
assert dupes == 0, "grain broken — P1 should have handled this"

# Target column present
assert "podium" in base.columns, "target 'podium' missing"
print("target dtype:", base["podium"].dtype)
print("target values:", base["podium"].unique())

# Target rate on the raw base
print("\nOverall podium rate:", base["podium"].mean().round(4))

In [ ]:
yearly = base.groupby("year")["podium"].agg(["mean", "count"])
print("Podium rate, last 10 years:")
print(yearly.tail(10))

print("\nPodium rate, first 10 years:")
print(yearly.head(10))

In [ ]:
miss = base.isna().mean().sort_values(ascending=False)
print("Top 15 columns by missing share:")
print(miss.head(15).round(4))

print("\nColumns with zero missing:")
print(miss[miss == 0].index.tolist())

### Section 3.3: Training start year

The ml table uses qualifying features (qual_position, q1_s, q2_s, q3_s,
gap_to_pole_pct). 

Qualifying data does not exist for most of F1 history (e.g. 1950-199x),
so we need to choose the earliest year to include in the ML table.

We look at qualifying coverage per season and pick the earliest year where it's stable.

In [ ]:
# Calendar table — needed to bring year/round into standings and results
races = tables["races"][["raceId", "year", "round", "circuitId", "date"]].copy()
races.head()

In [ ]:
# Qualifying coverage per season: share of rows with a q1_s time
cov = base.groupby("year")["has_qualifying"].mean()

# Also look at q1_s, q2_s, q3_s separately
q_cov = base.groupby("year")[["q1_s", "q2_s", "q3_s"]].apply(lambda d: d.notna().mean())

print("has_qualifying per year (first 10 and last 10):")
print(cov.head(10))
print(cov.tail(10))

print("\nq1_s / q2_s / q3_s coverage for 1994-2010:")
print(q_cov.loc[1994:2010])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cov.index, cov.values, marker="o", label="has_qualifying")
ax.axvline(2003, color="red", linestyle="--", label="START_YEAR = 2003")

ax.set_title("Qualifying coverage per season")
ax.set_xlabel("Year")
ax.set_ylabel("Share of rows with qualifying data")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

We chose 2003 since it defines the start of a stable inclusion of qualifying data. 
Despite q2 and q3 missing for some of the years after and including 2003, we can handle it with reached_q2, reached_q3, and qual_format rather than imputation. 

In [ ]:
START_YEAR = 2003
model = base[base["year"] >= START_YEAR].copy()
model = model.sort_values(["year", "round", "raceId", "driverId"]).reset_index(drop=True)

print("Rows kept:", model.shape[0])
print("Years:", model["year"].min(), "-", model["year"].max())
print("Rows dropped:", base.shape[0] - model.shape[0])

In [ ]:
model["reached_q2"]  = model["q2_s"].notna().astype(int)
model["reached_q3"]  = model["q3_s"].notna().astype(int)
model["qual_format"] = (model["year"] >= 2006).astype(int)

model[["year", "q1_s", "q2_s", "q3_s", "reached_q2", "reached_q3", "qual_format"]]
#qual_format tells us if that year was before the knockout era where the q2 and q3 
#were getting recorded starting from 2006
#we're gonna use it to see the reason q2 and q3 are missing within the data

In [ ]:
print("model shape:", model.shape)

In [ ]:
print("model shape:", model.shape)
print("model years:", model["year"].min(), "-", model["year"].max())

### Section 3.4: Grid and qualifying features

The prediction cutoff is after qualifying, and before the race. This means all the following columns are safe for our predicitons.

**Grid features**:
- `grid` is the actual starting position after any penalties, (0 = no grid slot, handled by pit_lane_flag).
- `pit_lane_flag` is 1 if the driver started from the pit lane where `grid==0`, it is 0 otherwise.

**Qualifying features**
- `qual_position` is the driver's rank in the qualifying session
- `best_qual_s` is the driver's best qualifying lap in seconds
  (`min(q1_s, q2_s, q3_s)`).
- `gap_to_pole_pct` is the percentage of how much slower a driver was than the pole sitter
  in the same race (the fastest driver). Per-race normalised, so comparable across eras.

**Missingness.** Qualifying data does not exist for most of F1 history, so
we cut the ML table at 2003 (see 3.3). From 2006 onward, `q2_s` and `q3_s`
are also structurally missing for drivers eliminated in earlier rounds;
we handle this with `reached_q2`, `reached_q3`, and `qual_format` rather
than imputing.  

** All are set during or before grid formation. None come
from the race itself.**

In [ ]:
# Grid features
model["pit_lane_flag"] = model["pitlane_start"].astype(int)

print("Grid feature summary:")
print(model[["grid", "pit_lane_flag"]].describe())
print("\nPit-lane starts share:", model["pit_lane_flag"].mean().round(4))
print("Rows with grid == 0:", (model["grid"] == 0).sum())

In [ ]:
model["best_qual_s"] = model[["q1_s", "q2_s", "q3_s"]].min(axis=1)

pole = model.groupby("raceId")["best_qual_s"].transform("min")
model["gap_to_pole_pct"] = (model["best_qual_s"] - pole) / pole * 100

print(model[["grid", "pit_lane_flag", "qual_position",
             "best_qual_s", "gap_to_pole_pct"]].describe())

In [ ]:
FORBIDDEN = {"positionOrder", "position", "positionText", "points", "laps",
             "time", "milliseconds", "fastestLap", "rank", "status"}
used = {"grid", "pit_lane_flag", "qual_position", "best_qual_s", "gap_to_pole_pct"}
assert not (used & FORBIDDEN), "post-race column used as feature"
print("grid + qualifying leakage check OK")

### Section 3.5: Pre-race standings ###

The standings tables (`driver_standings`, `constructor_standings`) record
cumulative points and rank **after** each race. Using them directly would
leak the current race's result into the features.

Fix: merge `races` to bring `year` and `round` into the standings, sort
by season+round, then `groupby(id).shift(1)` so each row holds the
standings **after the previous round** = standings going into this race.

**Leakage assertion.** Round 1 of every season must have NaN pre-race
standings (there is no previous race). The assertion fails loudly if
this is violated.

**Missingness.** ~5–7% of rows (round 1 of each season). We do NOT impute.

In [ ]:
# Bring year/round in — needed to sort standings chronologically
races = tables["races"][["raceId", "year", "round"]].copy()
print(races.shape)
races.head()

In [ ]:
ds = tables["driver_standings"].copy()
ds = ds.merge(races, on="raceId", how="left")
ds = ds.sort_values(["year", "round", "driverId"])

ds["pts_before"] = ds.groupby(["year", "driverId"])["points"].shift(1)
ds["pos_before"] = ds.groupby(["year", "driverId"])["position"].shift(1)

ds_pre = ds[["raceId", "driverId", "pts_before", "pos_before"]].copy()
ds_pre.head()

In [ ]:
# What year/round is raceId 833?
print(races[races["raceId"] == 833])

# Look at more rows to see the shift working
# (filter to one driver across a season)
sample_driver = ds_pre["driverId"].iloc[0]
print("\nSample driver:", sample_driver)
sample = ds_pre.merge(races, on="raceId").query("driverId == @sample_driver")
print(sample.sort_values(["year", "round"]).head(10))

In [ ]:
# Merge pre-race driver standings into the modeling table
model = model.merge(ds_pre, on=["raceId", "driverId"], how="left")

print("model shape:", model.shape)
print(model[["pts_before", "pos_before"]].isna().mean())

# Assertion: first race of each season must have NaN pre-race standings
first_race = model.groupby("year")["round"].transform("min") == model["round"]
assert model.loc[first_race, "pts_before"].isna().all(), "LEAK: standings before round 1"
assert model.loc[first_race, "pos_before"].isna().all(), "LEAK: standings before round 1"
print("driver standings leakage check OK")

In [ ]:
# --- Constructor standings: standings going INTO each race ---
cs = tables["constructor_standings"].copy()
cs = cs.merge(races, on="raceId", how="left")
cs = cs.sort_values(["year", "round", "constructorId"])

cs["cons_pts_before"] = cs.groupby(["year", "constructorId"])["points"].shift(1)
cs["cons_pos_before"] = cs.groupby(["year", "constructorId"])["position"].shift(1)

cs_pre = cs[["raceId", "constructorId", "cons_pts_before", "cons_pos_before"]].copy()
cs_pre.head()

In [ ]:
print(races[races["raceId"].isin([765, 766])])

In [ ]:
# Merge into model
model = model.merge(cs_pre, on=["raceId", "constructorId"], how="left")

print("model shape:", model.shape)
print(model[["cons_pts_before", "cons_pos_before"]].isna().mean())

# Assertion: round 1 of each season must have NaN
assert model.loc[first_race, "cons_pts_before"].isna().all(), "LEAK: constructor standings"
assert model.loc[first_race, "cons_pos_before"].isna().all(), "LEAK: constructor standings"
print("constructor standings leakage check OK")

### 3.6 Rolling form ###

We build a 5-race rolling podium rate for each driver and each constructor:
"how often did this driver/team podium in the last 5 races?"

The rule is that rolling windows must use only past races. We enforce this
with `shift(1)` before `.rolling()`: row N never sees its own race's result.

For constructors, we first collapse to one row per team-race, because two
drivers share the same constructor. Naive `groupby("constructorId").shift(1)`
would leak the teammate's same-race result into the form value.

Leakage assertions:
- First career race per driver = NaN
- First race per constructor = NaN
- Both drivers of a team-race share the same constructor_form5 value

In [ ]:
# Clean slate: drop the old form columns if they exist
model = model.drop(columns=["driver_form5", "constructor_form5"], errors="ignore")
print("model columns now:", model.shape)

In [ ]:
# Compute form on ALL years so pre-2003 history is available
base_sorted = base.sort_values(["driverId", "year", "round"]).reset_index(drop=True)

base_sorted["driver_form5"] = (
    base_sorted.groupby("driverId")["podium"]
               .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
)

# Merge onto model (2003+)
model = model.merge(
    base_sorted[["raceId", "driverId", "driver_form5"]],
    on=["raceId", "driverId"],
    how="left"
)

model[["driverId", "year", "round", "podium", "driver_form5"]].head(10)

In [ ]:
# First career race ever (from base) must be NaN — check by looking at base_sorted
first_driver_career = base_sorted.groupby("driverId").cumcount() == 0
assert base_sorted.loc[first_driver_career, "driver_form5"].isna().all(), "LEAK: driver form"

# Also check: within model, no NaN except genuinely-new drivers
print("driver_form5 NaN share in model:", model["driver_form5"].isna().mean().round(4))
print("driver form leakage check OK")

In [ ]:
# Collapse to one row per (constructor, race) on ALL years
base_cons = (
    base.groupby(["constructorId", "raceId", "year", "round"], as_index=False)["podium"]
        .mean()
)

base_cons = base_cons.sort_values(["constructorId", "year", "round"])
base_cons["constructor_form5"] = (
    base_cons.groupby("constructorId")["podium"]
             .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
)

# Merge onto model
model = model.merge(
    base_cons[["raceId", "constructorId", "constructor_form5"]],
    on=["raceId", "constructorId"],
    how="left"
)

model[["constructorId", "year", "round", "podium", "constructor_form5"]].head(10)

In [ ]:
# 1. First race per constructor (chronologically) must be NaN
first_per_constructor = (
    base_cons.sort_values(["constructorId", "year", "round"])
            .groupby("constructorId")
            .head(1)
)
assert first_per_constructor["constructor_form5"].isna().all(), \
    "LEAK: constructor form on first race"

# 2. Both drivers of a team-race must share the same value
grouped = model.groupby(["raceId", "constructorId"])["constructor_form5"].nunique(dropna=False)
assert (grouped <= 1).all(), \
    "constructor form differs within the same team-race"

print("constructor form leakage checks OK")
print("constructor_form5 NaN share in model:", model["constructor_form5"].isna().mean().round(4))

In [ ]:
# Put rows back in chronological race order for downstream sections
model = model.sort_values(["year", "round", "raceId", "driverId"]).reset_index(drop=True)

print("model shape:", model.shape)
model[["raceId", "driverId", "year", "round",
       "driver_form5", "constructor_form5", "podium"]].head(5)

### Section 3.7: Circuit history + home flag + era ###

we have 3 feature groups to add. 

**Circuit history.** Some drivers are specialists at certain tracks (e.g. Senna
at Monaco, Hamilton at Silverstone). We capture this with two features per
(driver, circuit):
- `circuit_starts_before` — number of previous races by this driver at this circuit
- `circuit_podiums_before` — number of previous podiums by this driver at this circuit

Both use only past races. The current race is excluded by `shift(1).cumsum()`.

We compute on the full `base` table (all years), then merge onto `model`
(2003+) so 2003 rows have their pre-2003 circuit history.

**Home flag.** `home_flag = 1` when the driver's nationality matches the
circuit's country (e.g. British driver at Silverstone). Uses P2's
nationality → country mapping. [Placeholder until P2 sends it.]

**Era.** F1 rules and car performance shift by technical era. We bucket
years: `pre_2014` (2003–2013), `hybrid` (2014–2021), `ground_effect` (2022+).
Year is known before the race.

In [ ]:
base_sorted_circuit = base.sort_values(["driverId", "circuitId", "year", "round"]).reset_index(drop=True)

base_sorted_circuit["circuit_starts_before"] = (
    base_sorted_circuit.groupby(["driverId", "circuitId"]).cumcount()
)

base_sorted_circuit["circuit_podiums_before"] = (
    base_sorted_circuit.groupby(["driverId", "circuitId"])["podium"]
                       .transform(lambda s: s.shift(1).fillna(0).cumsum())
)

# Re-merge onto model (drop old columns first to avoid suffix mess)
model = model.drop(columns=["circuit_starts_before", "circuit_podiums_before"], errors="ignore")
model = model.merge(
    base_sorted_circuit[["raceId", "driverId",
                         "circuit_starts_before", "circuit_podiums_before"]],
    on=["raceId", "driverId"],
    how="left"
)

model[["driverId", "circuitId", "year", "round",
       "circuit_starts_before", "circuit_podiums_before", "podium"]].head(10)

In [ ]:
# Assertion on base_sorted_circuit (all years) — not on model
first_at_circuit_base = (
    base_sorted_circuit.sort_values(["driverId", "circuitId", "year", "round"])
                      .groupby(["driverId", "circuitId"])
                      .cumcount() == 0
)
assert (base_sorted_circuit.loc[first_at_circuit_base, "circuit_starts_before"] == 0).all(), \
    "LEAK: starts before"
assert (base_sorted_circuit.loc[first_at_circuit_base, "circuit_podiums_before"] == 0).all(), \
    "LEAK: podiums before"
print("circuit history leakage check OK")

In [ ]:
first = base_sorted_circuit.loc[first_at_circuit_base]
print("Total first-occurrence rows:", len(first))
print("Non-zero podiums:", (first["circuit_podiums_before"] != 0).sum())
print("NaN podiums:", first["circuit_podiums_before"].isna().sum())
print("\nSample:")
print(first[["driverId", "circuitId", "year", "circuit_podiums_before"]].head(10))

In [ ]:
# --- P2's authoritative mapping (from their Q2 notebook) ---
NATIONALITY_TO_COUNTRY = {
    "American": "USA", "American-Italian": "USA", "Argentine": "Argentina",
    "Argentine-Italian": "Argentina", "Argentinian": "Argentina", "Australian": "Australia",
    "Austrian": "Austria", "Belgian": "Belgium", "Brazilian": "Brazil", "British": "UK",
    "Canadian": "Canada", "Chilean": "Chile", "Chinese": "China", "Colombian": "Colombia",
    "Czech": "Czech Republic", "Danish": "Denmark", "Dutch": "Netherlands",
    "East German": "Germany", "Finnish": "Finland", "French": "France", "German": "Germany",
    "Hungarian": "Hungary", "Indian": "India", "Indonesian": "Indonesia", "Irish": "Ireland",
    "Italian": "Italy", "Japanese": "Japan", "Liechtensteiner": "Liechtenstein",
    "Malaysian": "Malaysia", "Mexican": "Mexico", "Monegasque": "Monaco",
    "New Zealander": "New Zealand", "Polish": "Poland", "Portuguese": "Portugal",
    "Rhodesian": "Rhodesia", "Russian": "Russia", "South African": "South Africa",
    "Spanish": "Spain", "Swedish": "Sweden", "Swiss": "Switzerland", "Thai": "Thailand",
    "Uruguayan": "Uruguay", "Venezuelan": "Venezuela",
}
SECOND_COUNTRY = {"American-Italian": "Italy", "Argentine-Italian": "Italy"}

# --- Drivers: nationality, stripped ---
drivers_nat = tables["drivers"][["driverId", "nationality"]].copy()
drivers_nat["nationality"] = drivers_nat["nationality"].str.strip()

# --- Circuits: country, normalized ---
circuits_country = tables["circuits"][["circuitId", "country"]].copy()
circuits_country["country"] = circuits_country["country"].replace("United States", "USA")

# --- Merge onto model ---
model = model.merge(drivers_nat, on="driverId", how="left", validate="many_to_one")
model = model.merge(circuits_country, on="circuitId", how="left", validate="many_to_one")

# --- home_flag: circuit country matches driver's primary OR secondary home country ---
model["home_country"] = model["nationality"].map(NATIONALITY_TO_COUNTRY)
model["second_home_country"] = model["nationality"].map(SECOND_COUNTRY)

model["home_flag"] = (
    (model["country"] == model["home_country"]) |
    (model["country"] == model["second_home_country"])
).astype(int)

# --- Sanity checks (mirroring P2's asserts) ---
assert model["nationality"].notna().all(), "driver missing nationality"
assert model["country"].notna().all(), "circuit missing country"
unmapped = set(model["nationality"]) - set(NATIONALITY_TO_COUNTRY)
assert len(unmapped) == 0, f"nationalities missing from mapping: {unmapped}"

print("home_flag value counts:")
print(model["home_flag"].value_counts())
print("\nHome starts per country (top 10):")
print(model[model["home_flag"] == 1]["country"].value_counts().head(10))

In [ ]:
# Keep only home_flag — drop helper columns from the join
model = model.drop(columns=["nationality", "country", "home_country", "second_home_country"],
                   errors="ignore")
print("model shape:", model.shape)

In [ ]:
print("Columns in model:")
for i, col in enumerate(sorted(model.columns), 1):
    print(f"{i:2d}. {col}")
print("\nTotal:", len(model.columns))

In [ ]:
import pandas as pd
model["era"] = pd.cut(
    model["year"],
    bins=[2002, 2013, 2021, 2024],
    labels=["v8", "hybrid", "ground_effect"]
)
print(model["era"].value_counts())
print("\nmodel shape:", model.shape)

In [ ]:
print("Columns in model:")
for i, col in enumerate(sorted(model.columns), 1):
    print(f"{i:2d}. {col}")
print("\nTotal:", len(model.columns))

### Section 3.8: Data dictionary

The modeling table has 33 columns: 16 from P1 (keys + target + grid/qual raw), 
and 18 features grouped into 6 categories for the ablation.


**Note on identifiers.** `driverId`, `constructorId`, `circuitId`, and `number` 
are not in FEATURES — they are identifiers, not features. Their signal is 
captured by the historical aggregates (driver_form5, constructor_form5, 
circuit_starts_before, circuit_podiums_before).

**Note on pit_lane_flag.** This is the standardised int version of P1's raw 
`pitlane_start`. Only `pit_lane_flag` is in FEATURES; the raw column is not used.

In [ ]:
FEATURE_GROUPS = {
    "grid":       ["grid", "pit_lane_flag"],
    "qualifying": ["qual_position", "best_qual_s", "gap_to_pole_pct",
                   "reached_q2", "reached_q3", "qual_format"],
    "standings":  ["pts_before", "pos_before",
                   "cons_pts_before", "cons_pos_before"],
    "form":       ["driver_form5", "constructor_form5"],
    "circuit":    ["circuit_starts_before", "circuit_podiums_before"],
    "context":    ["home_flag", "era"],
}
FEATURES = [f for group in FEATURE_GROUPS.values() for f in group]

print(f"Total features: {len(FEATURES)}")
for group, cols in FEATURE_GROUPS.items():
    print(f"  {group}: {len(cols)}")

In [ ]:
FORBIDDEN = {"positionOrder", "position", "positionText", "points", "laps",
             "time", "milliseconds", "fastestLap", "rank", "status"}
assert not (set(FEATURES) & FORBIDDEN), f"forbidden in FEATURES: {set(FEATURES) & FORBIDDEN}"
print("all 18 features are leakage-safe")

In [ ]:
import json
import os
os.makedirs("../reports", exist_ok=True)

with open("../reports/p3_feature_groups.json", "w") as f:
    json.dump(FEATURE_GROUPS, f, indent=2)

# Save as CSV instead of parquet
model.to_csv("../reports/p3_modeling_table.csv", index=False)

print("saved:")
print(" - reports/p3_feature_groups.json")
print(" - reports/p3_modeling_table.csv")
print(" - shape:", model.shape)

### Section 3.9: Evidence plots

will be done later

### Section 3.10: Time split ###

We will split the 2003-2024 rows into: 
- **train**: year ≤ 2019
- **validation**: year 2020–2021
- **test**: year ≥ 2022

**Why these years.** 2020–21 gives us two full seasons (with one shortened 
by COVID) for hyperparameter tuning. 2022+ is a clean, larger test set 
(3 seasons) that includes the new ground-effect era. Test set is used once.

In [ ]:
import numpy as np

model["split"] = np.select(
    [model["year"] <= 2019,
     model["year"].between(2020, 2021),
     model["year"] >= 2022],
    ["train", "validation", "test"],
    default="unknown"
)

print(model["split"].value_counts())
print("\nmodel shape:", model.shape)

In [ ]:
# Class balance per split
balance = model.groupby("split")["podium"].agg(["mean", "count"])
print("Class balance per split:")
print(balance.round(4))

# Plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
balance_ordered = balance.reindex(["train", "validation", "test"])
bars = ax.bar(balance_ordered.index, balance_ordered["mean"] * 100,
              color=["tab:blue", "tab:orange", "tab:green"])
ax.bar_label(bars, fmt="%.1f%%")
ax.set_title("Podium rate per split (class balance)")
ax.set_xlabel("Split")
ax.set_ylabel("Podium rate (%)")
ax.set_ylim(0, 20)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("../reports/fig_p3_split_balance.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
model.to_csv("../reports/p3_modeling_table.csv", index=False)
print("saved with split — shape:", model.shape)
print("columns:", len(model.columns))